In [6]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

In [7]:
df = pd.read_csv('marriage_longevity_master.csv')
df.head()

,marriage_id,marriage_number,age_at_marriage,age_gap_years,education_level,household_income_usd,financial_stress,both_employed,cohabited_before,premarital_counseling,...,contempt,defensiveness,stonewalling,repair_attempt_success,positive_negative_ratio,conflict_frequency_weekly,shared_activities_weekly,years_married,years_to_divorce,divorced
0,MAR000000,1,36,6,less_than_hs,68600.0,5.9,1,1,0,...,7.7,5.8,8.1,4.1,2.22,3.0,4.0,22.0,NaN,0
1,MAR000001,2,59,1,some_college,328800.0,0.0,1,1,0,...,1.7,6.8,3.0,5.1,11.99,1.0,1.0,15.5,NaN,0
2,MAR000002,2,44,1,high_school,139900.0,6.1,0,0,1,...,6.5,4.2,4.8,5.7,2.27,4.0,2.0,7.2,NaN,0
3,MAR000003,1,25,2,bachelors,97600.0,5.9,1,1,1,...,0.0,4.1,0.9,8.7,6.24,0.0,4.0,13.5,NaN,0
4,MAR000004,2,29,3,graduate,98200.0,4.0,0,0,1,...,4.4,4.6,9.6,8.9,2.59,4.0,5.0,7.5,7.5,1


In [8]:
df['total_negativity'] = (df['defensiveness'] + df['stonewalling'] + df['contempt'] + df['criticism'])
df['income_per_person'] = (df['household_income_usd']) / (df['n_children'] + 2)
df['conflict_x_repair'] = (df['conflict_frequency_weekly'] * df['repair_attempt_success'])

In [9]:
df_cleaned = df.copy()
df_cleaned.head()

,marriage_id,marriage_number,age_at_marriage,age_gap_years,education_level,household_income_usd,financial_stress,both_employed,cohabited_before,premarital_counseling,...,repair_attempt_success,positive_negative_ratio,conflict_frequency_weekly,shared_activities_weekly,years_married,years_to_divorce,divorced,total_negativity,income_per_person,conflict_x_repair
0,MAR000000,1,36,6,less_than_hs,68600.0,5.9,1,1,0,...,4.1,2.22,3.0,4.0,22.0,NaN,0,26.4,17150.000000,12.3
1,MAR000001,2,59,1,some_college,328800.0,0.0,1,1,0,...,5.1,11.99,1.0,1.0,15.5,NaN,0,11.7,65760.000000,5.1
2,MAR000002,2,44,1,high_school,139900.0,6.1,0,0,1,...,5.7,2.27,4.0,2.0,7.2,NaN,0,20.7,46633.333333,22.8
3,MAR000003,1,25,2,bachelors,97600.0,5.9,1,1,1,...,8.7,6.24,0.0,4.0,13.5,NaN,0,8.7,24400.000000,0.0
4,MAR000004,2,29,3,graduate,98200.0,4.0,0,0,1,...,8.9,2.59,4.0,5.0,7.5,7.5,1,24.0,24550.000000,35.6


In [10]:
df_cleaned = df_cleaned.drop(columns=['marriage_id', 'years_to_divorce', 'years_married'], axis=1) # 'years_married'


In [11]:
df_cleaned = pd.get_dummies(data=df_cleaned, columns=['education_level'], dtype=int, drop_first=True)
df_cleaned = pd.get_dummies(data=df_cleaned, columns=['religious_attendance'], dtype= int, drop_first= True)

In [12]:
X = df_cleaned.drop(columns=['divorced'], axis=1)
y = df_cleaned['divorced']

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [14]:

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
n_columns = [ 'conflict_x_repair','total_negativity','income_per_person', 'marriage_number', 'age_at_marriage', 'age_gap_years', 'household_income_usd', 'financial_stress', 'criticism',
             'contempt', 'defensiveness', 'stonewalling', 'repair_attempt_success', 'positive_negative_ratio',
             'conflict_frequency_weekly', 'shared_activities_weekly'] # 'conflict_x_repair','total_negativity','income_per_person',
X_train[n_columns] = scaler.fit_transform(X_train[n_columns])
X_test[n_columns] = scaler.transform(X_test[n_columns])

In [15]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()

In [16]:
X_train.columns

Index(['marriage_number', 'age_at_marriage', 'age_gap_years',
       'household_income_usd', 'financial_stress', 'both_employed',
       'cohabited_before', 'premarital_counseling', 'child_before_marriage',
       'n_children', 'criticism', 'contempt', 'defensiveness', 'stonewalling',
       'repair_attempt_success', 'positive_negative_ratio',
       'conflict_frequency_weekly', 'shared_activities_weekly',
       'total_negativity', 'income_per_person', 'conflict_x_repair',
       'education_level_graduate', 'education_level_high_school',
       'education_level_less_than_hs', 'education_level_some_college',
       'religious_attendance_never', 'religious_attendance_rarely',
       'religious_attendance_weekly'],
      dtype='object')

In [17]:
model.fit(X_train, y_train)

LogisticRegression()

In [18]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
y_pred = model.predict(X_test)

In [19]:
accuracy_score(y_test, y_pred)
# accuracy_score : 0.7534 (excluding years_married but did feature engineering)
# accuracy_score : 0.78511 (including years_married )
# accuracy_score : 0.7531 (excluding feature engineering and dropped years_to_married )

0.7535555555555555

In [20]:
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.76      0.79      0.78      4874
           1       0.74      0.71      0.72      4126

    accuracy                           0.75      9000
   macro avg       0.75      0.75      0.75      9000
weighted avg       0.75      0.75      0.75      9000

[[3870 1004]
 [1214 2912]]


In [21]:
from sklearn.neighbors import KNeighborsClassifier
model_knn = KNeighborsClassifier(n_neighbors=5)

In [22]:
model_knn.fit(X_train, y_train)

KNeighborsClassifier()

In [23]:
knn_y_pred = model_knn.predict(X_test)


In [24]:
accuracy_score(y_test, knn_y_pred)
print(classification_report(y_test, knn_y_pred))

              precision    recall  f1-score   support

           0       0.71      0.74      0.72      4874
           1       0.68      0.65      0.66      4126

    accuracy                           0.70      9000
   macro avg       0.69      0.69      0.69      9000
weighted avg       0.70      0.70      0.70      9000



In [25]:
from sklearn.naive_bayes import GaussianNB
model_nb = GaussianNB()

In [26]:
model_nb.fit(X_train, y_train)

GaussianNB()

In [27]:
nb_y_pred = model_nb.predict(X_test)

In [28]:
print(accuracy_score(y_test, nb_y_pred))
print(classification_report(y_test, nb_y_pred))

0.7292222222222222
              precision    recall  f1-score   support

           0       0.76      0.73      0.74      4874
           1       0.69      0.73      0.71      4126

    accuracy                           0.73      9000
   macro avg       0.73      0.73      0.73      9000
weighted avg       0.73      0.73      0.73      9000



In [29]:
from sklearn.tree import DecisionTreeClassifier
model_dt = DecisionTreeClassifier()

In [30]:
model_dt.fit(X_train, y_train)

DecisionTreeClassifier()

In [31]:
dt_y_pred = model_dt.predict(X_test)

In [32]:
print(accuracy_score(y_test, dt_y_pred))
print(classification_report(y_test, dt_y_pred))

0.6418888888888888
              precision    recall  f1-score   support

           0       0.67      0.68      0.67      4874
           1       0.61      0.60      0.61      4126

    accuracy                           0.64      9000
   macro avg       0.64      0.64      0.64      9000
weighted avg       0.64      0.64      0.64      9000



In [33]:
from sklearn.svm import SVC
model_svm = SVC()

In [34]:
model_svm.fit(X_train, y_train)

SVC()

In [35]:
svm_y_predict = model_svm.predict(X_test)

In [36]:
print(accuracy_score(y_test, svm_y_predict))
print(classification_report(y_test, svm_y_predict))

0.7493333333333333
              precision    recall  f1-score   support

           0       0.76      0.79      0.77      4874
           1       0.74      0.70      0.72      4126

    accuracy                           0.75      9000
   macro avg       0.75      0.75      0.75      9000
weighted avg       0.75      0.75      0.75      9000



In [37]:
comparison = pd.DataFrame({
    "actual": y_test,
    "LR": y_pred,
    "KNN": knn_y_pred,
    "SVC": svm_y_predict
})

comparison["LR_correct"] = comparison["LR"] == comparison["actual"]
comparison["KNN_correct"] = comparison["KNN"] == comparison["actual"]
comparison["SVC_correct"] = comparison["SVC"] == comparison["actual"]

comparison.sample(20)

,actual,LR,KNN,SVC,LR_correct,KNN_correct,SVC_correct
27662,0,0,0,0,True,True,True
37319,1,0,1,1,False,True,True
38801,0,0,0,0,True,True,True
20512,0,0,0,0,True,True,True
13814,0,0,0,0,True,True,True
10421,1,1,1,1,True,True,True
9736,0,1,1,0,False,False,True
35957,0,1,1,1,False,False,False
11449,1,0,0,1,False,False,True
29030,1,0,0,0,False,False,False


In [38]:
# while choosing base learner we have to remember a couple of things to get best possible results
# 1. reasonably good + make difference mistakes.
# 2. don't choose terrible model just for diversity. it doesn't mean 40% accuracy is good
(y_pred != knn_y_pred).mean()

np.float64(0.20122222222222222)

In [ ]:
from sklearn.ensemble import StackingClassifier
base_learner = [('nb', GaussianNB()), ('knn', KNeighborsClassifier(n_neighbors=5)), ('svc', SVC(kernel='rbf', random_state=42))]
meta_learner = LogisticRegression()
stacking_clf = StackingClassifier(estimators=base_learner, final_estimator=meta_learner)

In [ ]:
stacking_clf.fit(X_train, y_train)

In [ ]:
stacking_y_pred = stacking_clf.predict(X_test)

In [ ]:
accuracy_score(y_test, stacking_y_pred)
# 0.74911

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)

In [ ]:
rf.fit(X_train, y_train)

In [ ]:
rf_y_pred = rf.predict(X_test)

In [ ]:
accuracy_score(y_test, rf_y_pred)
# accuracy : 0.736

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators= 100, random_state=42)

In [ ]:
gb.fit(X_train, y_train)

In [ ]:
gb_y_pred = gb.predict(X_test)

In [ ]:
accuracy_score(y_test, gb_y_pred)
# 0.74.91

In [ ]:
from xgboost import XGBClassifier
xgb = XGBClassifier(n_estimators = 100, learning_rate = 0.1, max_depth = 3, random_state = 42, eval_metric = 'logloss')

In [ ]:
xgb.fit(X_train, y_train)

In [ ]:
xgb_y_pred = xgb.predict(X_test)


In [ ]:
accuracy_score(y_test, xgb_y_pred)
# 0.7493

In [ ]:
y.value_counts(normalize=True)

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
results = []
models = [model, model_svm, model_dt, model_nb, model_knn]
for mod in models:
    scores = cross_val_score(mod, X_train, y_train, cv=kfold, scoring = 'accuracy')
    results.append({'Model' : mod,
                    'Mean Accuracy' : scores.mean(),
                    'Std' : scores.std()})
results_df = pd.DataFrame(results)
print(results_df.sort_values(by='Mean Accuracy', ascending=False))

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {'C' : [0.0, 0.1, 10, 100], 'solver' : ['lbfgs', 'liblinear']}
grid_lr = GridSearchCV(LogisticRegression(max_iter= 1000), param_grid, cv = kfold, scoring='accuracy', n_jobs=-1)

In [ ]:
grid_lr.fit(X_train, y_train)

In [ ]:
print(grid_lr.best_params_)
print(grid_lr.best_score_)

In [ ]:
# param_grid_svc = {'C' : [0.1, 1, 10, 100], 'gamma' : ['scale', 'auto', 0.01, 0.1, 1]}
# grid_svc = GridSearchCV(SVC(kernel='rbf'), param_grid_svc, cv=kfold, scoring='accuracy', n_jobs=-1)
# grid_svc.fit(X_train, y_train)

In [ ]:
# print(grid_svc.best_params_)
# print(grid_svc.best_score_)

# didn't run due to high computational cost